# Logistic Regression — Loss Function, Cost Function, and Optimization

## 1. Introduction

Logistic regression is a supervised machine learning algorithm used primarily for binary classification problems. Unlike linear regression, which predicts continuous numerical values, logistic regression estimates the probability that an observation belongs to a particular class.

Examples of binary classification include:

- Predicting whether a vehicle component will fail a diagnostic test.
- Classifying an email as spam or not spam.
- Predicting whether a customer will leave a service.
- Determining whether a sensor measurement indicates a potentially unsafe condition.

Logistic regression produces a probability between 0 and 1 using the sigmoid function:

\[
f_{w,b}(x)=\frac{1}{1+e^{-(w^Tx+b)}}
\]

Where:

- \(x\) represents the input features.
- \(w\) represents the model weights.
- \(b\) is the bias or intercept.
- \(f_{w,b}(x)\) is the predicted probability that the output belongs to class 1.

For example, a model might predict a 0.85 probability that a vehicle component will fail a test. A classification threshold, often 0.5, can then convert this probability into a predicted class.

However, producing a probability is only part of the learning process. The model must also measure how accurate its predictions are and adjust its parameters to improve them. This is the purpose of the **loss function and cost function**.

## 2. Understanding Loss and Cost

Although these terms are closely related, they describe different levels of the learning process.

### Loss function

The loss function measures the error associated with a single training example. It compares the true label with the probability predicted by the model.

For a single example:

\[
L(f(x),y)
\]

Where \(y\) is the true label and \(f(x)\) is the predicted probability.

### Cost function

The cost function aggregates the losses from the training dataset into a single value. In the standard formulation, it calculates the average loss across all \(m\) training examples.

\[
J(w,b)=\frac{1}{m}\sum_{i=1}^{m}L(f_{w,b}(x^{(i)}),y^{(i)})
\]

The distinction is important:

- **Loss** evaluates an individual prediction.
- **Cost** evaluates the model's overall performance on the training dataset.

During training, gradient descent uses the cost function to determine how the model's weights and bias should change.

## 3. Why Squared Error Is Not Suitable for Logistic Regression

Squared error is commonly used in linear regression because the model predicts continuous values and the resulting cost function has convenient mathematical properties.

For a single example, squared error can be expressed as:

\[
L(f(x),y)=\frac{1}{2}(f(x)-y)^2
\]

The factor of \(1/2\) is included to simplify differentiation.

In linear regression, the prediction is a linear function of the input features. Consequently, the squared error cost function is convex with respect to the model parameters.

Logistic regression, however, uses the sigmoid function to transform a linear combination of features into a probability:

\[
f_{w,b}(x)=\sigma(w^Tx+b)
\]

Combining this nonlinear sigmoid function with squared error can produce a **non-convex cost function** with respect to the parameters.

### What does non-convex mean?

A convex cost function has a bowl-like shape, meaning that any local minimum is also a global minimum. A non-convex function can have a more complicated surface, potentially containing multiple local minima, saddle points, and regions with varying curvature.

This distinction matters because gradient descent updates the model parameters according to the gradient of the cost function:

\[
w_j := w_j-\alpha\frac{\partial J}{\partial w_j}
\]

Where:

- \(w_j\) is a model weight.
- \(\alpha\) is the learning rate.
- \(\frac{\partial J}{\partial w_j}\) is the partial derivative of the cost with respect to that weight.

With a non-convex cost function, the optimization process can become more difficult. Gradient descent may converge to a local minimum rather than the global minimum, depending on the cost surface and initialization.

This does not mean squared error can never be used with a sigmoid model. Rather, it is generally not the preferred objective for standard logistic regression because a more appropriate loss function provides better optimization properties and a sound statistical interpretation.

## 4. The Logistic Loss Function

Logistic regression uses a loss function based on the logarithm of the predicted probability assigned to the correct class.

This function is commonly called **log loss**, **logistic loss**, or **binary cross-entropy**.

For a single training example, its definition is:

\[
L(f(x),y)=
\begin{cases}
-\log(f(x)), & y=1\\
-\log(1-f(x)), & y=0
\end{cases}
\]

The loss depends on the true label because the model must assign a high probability to the correct class.

### Case 1: The true label is 1

When \(y=1\), the loss becomes:

\[
L(f(x),1)=-\log(f(x))
\]

The model should predict a probability close to 1.

| Predicted probability | Loss (approximately) | Interpretation |
|---:|---:|---|
| 0.99 | 0.010 | Very accurate, confident prediction |
| 0.90 | 0.105 | Good prediction |
| 0.70 | 0.357 | Reasonable prediction |
| 0.50 | 0.693 | Uncertain prediction |
| 0.10 | 2.303 | Incorrect, confident prediction |
| 0.01 | 4.605 | Extremely confident incorrect prediction |

As the predicted probability approaches 1, the loss approaches 0. As it approaches 0, the loss grows without bound.

The model is therefore rewarded for assigning a high probability to the true positive class and penalized for assigning a low probability to it.

### Case 2: The true label is 0

When \(y=0\), the loss becomes:

\[
L(f(x),0)=-\log(1-f(x))
\]

The model should predict a probability close to 0 for class 1.

| Predicted probability | Loss (approximately) | Interpretation |
|---:|---:|---|
| 0.01 | 0.010 | Very accurate, confident prediction |
| 0.10 | 0.105 | Good prediction |
| 0.30 | 0.357 | Reasonable prediction |
| 0.50 | 0.693 | Uncertain prediction |
| 0.90 | 2.303 | Incorrect, confident prediction |
| 0.99 | 4.605 | Extremely confident incorrect prediction |

Here, the term \(1-f(x)\) represents the predicted probability of class 0.

As the predicted probability of class 1 approaches 0, the probability assigned to class 0 approaches 1, and the loss approaches 0. Conversely, as the model becomes increasingly confident that the example belongs to class 1 when the true label is 0, the loss increases sharply.

### Why penalize confident incorrect predictions?

Consider a model that predicts whether a vehicle's braking system passes a verification test.

Suppose the true label is 1, meaning the component passes.

- A predicted probability of 0.60 indicates moderate confidence in the correct class.
- A predicted probability of 0.95 indicates much stronger confidence in the correct class.
- A predicted probability of 0.02 indicates strong confidence in the wrong class.

Log loss distinguishes these predictions by assigning progressively lower loss to better probability estimates and much higher loss to confident mistakes.

This encourages the model to learn parameters that assign high probabilities to the correct labels, rather than merely producing the correct class occasionally.

## 5. Simplifying the Loss Function

The two cases can be combined into a single mathematical expression:

\[
\boxed{
L(f(x),y)=-y\log(f(x))-(1-y)\log(1-f(x))
}
\]

This formulation works because the true label \(y\) is binary.

### When \(y=1\)

Substitute \(y=1\):

\[
\begin{aligned}
L(f(x),1)
&=-1\log(f(x))\\
&\quad -(1-1)\log(1-f(x))\\
&=-\log(f(x))
\end{aligned}
\]

The second term disappears because it is multiplied by zero.

### When \(y=0\)

Substitute \(y=0\):

\[
\begin{aligned}
L(f(x),0)
&=-0\log(f(x))\\
&\quad -(1-0)\log(1-f(x))\\
&=-\log(1-f(x))
\end{aligned}
\]

The first term disappears, leaving the loss for a negative example.

This is an example of using the label itself to select the appropriate mathematical expression without explicitly writing a conditional statement in the implementation.

It also makes the loss function convenient to vectorize, which is especially useful when training models on large datasets.

## 6. Deriving the Cost Function

The loss function evaluates one example, but machine learning models are trained on datasets containing many examples.

Let the training dataset contain \(m\) examples:

\[
\{(x^{(1)},y^{(1)}),\ldots,(x^{(m)},y^{(m)})\}
\]

Each example has its own input features and true binary label. The model produces a predicted probability for each example, and the loss function measures how well each prediction matches its label.

The cost function averages the individual losses:

\[
J(w,b)=\frac{1}{m}\sum_{i=1}^{m}L(f_{w,b}(x^{(i)}),y^{(i)})
\]

Substituting the simplified logistic loss gives:

\[
\boxed{
J(w,b)=-\frac{1}{m}\sum_{i=1}^{m}
\left[
y^{(i)}\log(f_{w,b}(x^{(i)}))
+(1-y^{(i)})\log(1-f_{w,b}(x^{(i)}))
\right]
}
\]

This is the standard binary cross-entropy cost function for logistic regression without regularization.

The summation adds the loss from every training example, while division by \(m\) calculates the average. The negative sign ensures that the resulting loss is nonnegative for valid predicted probabilities and becomes smaller as the model assigns higher probabilities to the correct labels.

### Example with a small dataset

Suppose a model evaluates four examples:

| Example | True label | Predicted probability of class 1 | Individual loss (approx.) |
|---|---:|---:|---:|
| 1 | 1 | 0.90 | 0.105 |
| 2 | 0 | 0.20 | 0.223 |
| 3 | 1 | 0.70 | 0.357 |
| 4 | 0 | 0.80 | 1.609 |

The average cost is:

\[
J=\frac{0.105+0.223+0.357+1.609}{4}
\]

\[
J\approx 0.574
\]

The fourth example has the largest loss because the model assigns a probability of 0.80 to class 1 even though its true label is 0.

If the model improves its predictions while keeping the other examples unchanged, the total cost will decrease. This gives the optimization algorithm a numerical objective to minimize.

## 7. Why Is the Logistic Regression Cost Function Convex?

Convexity is one of the key mathematical advantages of using logistic loss in standard logistic regression.

The sigmoid function itself is nonlinear. However, when logistic loss is applied to the sigmoid model, the resulting cost function is convex with respect to the weights and bias for ordinary logistic regression.

For an individual example, let:

\[
z=w^Tx+b
\]

and:

\[
f(x)=\sigma(z)
\]

The logistic loss can also be written as:

\[
L(z,y)=\log(1+e^z)-yz
\]

This equivalent form is useful for mathematical analysis and numerical implementation.

The second derivative of the loss with respect to \(z\) is:

\[
\frac{\partial^2 L}{\partial z^2}
=f(x)(1-f(x))
\]

Since \(0<f(x)<1\), the product \(f(x)(1-f(x))\) is nonnegative. This establishes that the loss is convex with respect to the linear score \(z\).

Because \(z\) is an affine function of the model parameters, the logistic loss is also convex with respect to those parameters. Summing convex losses and dividing by a positive constant preserves convexity, so the overall cost function remains convex.

### What does this mean for gradient descent?

For a differentiable convex cost function, every local minimum is a global minimum.

Therefore, gradient descent does not have to navigate between competing local minima in standard logistic regression. With an appropriate learning rate and suitable optimization conditions, it can converge toward a global minimum.

There are still practical challenges:

- An excessively large learning rate can cause unstable updates or divergence.
- An excessively small learning rate can make training very slow.
- Highly correlated features or poorly scaled inputs can slow convergence.
- If the dataset is perfectly separable, unregularized logistic regression may not have a finite parameter vector that attains its infimum.
- Different parameter vectors can sometimes achieve the same minimum, especially when features are redundant.

Thus, convexity makes optimization more reliable, but it does not guarantee that every training configuration will converge successfully.

## 8. Connection to Maximum Likelihood Estimation

Logistic loss is not an arbitrary mathematical choice. It follows naturally from the probabilistic assumptions of logistic regression.

The model assumes that the target \(y\) follows a Bernoulli distribution with probability \(f(x)\) of belonging to class 1:

\[
P(y\mid x)=f(x)^y(1-f(x))^{1-y}
\]

This expression handles both possible labels:

- If \(y=1\), the probability is \(f(x)\).
- If \(y=0\), the probability is \(1-f(x)\).

For a dataset of independent training examples, the likelihood is the product of the probabilities assigned to their observed labels:

\[
\mathcal{L}(w,b)=
\prod_{i=1}^{m}
f_{w,b}(x^{(i)})^{y^{(i)}}
(1-f_{w,b}(x^{(i)}))^{1-y^{(i)}}
\]

Maximum likelihood estimation seeks the parameters that maximize this likelihood.

Products of many probabilities can become numerically very small, so it is more convenient to take the logarithm. Since the logarithm is strictly increasing, maximizing the likelihood is equivalent to maximizing its log-likelihood.

Taking the negative average log-likelihood produces the logistic regression cost function:

\[
J(w,b)=-\frac{1}{m}\log\mathcal{L}(w,b)
\]

Expanding this expression gives the binary cross-entropy cost introduced earlier.

Therefore, **minimizing logistic regression cost is equivalent to maximizing the likelihood of the observed training labels**.

This connection gives the loss function a clear statistical interpretation: the model is searching for parameters that make the observed labels as probable as possible under its predicted probability distribution.

## 9. How Gradient Descent Uses the Cost Function

Once the cost function is defined, gradient descent can optimize the model parameters.

For logistic regression, the prediction for example \(i\) is:

\[
f^{(i)}=\sigma(w^Tx^{(i)}+b)
\]

The partial derivatives of the average cost are:

\[
\frac{\partial J}{\partial w_j}
=\frac{1}{m}\sum_{i=1}^{m}
(f^{(i)}-y^{(i)})x_j^{(i)}
\]

and:

\[
\frac{\partial J}{\partial b}
=\frac{1}{m}\sum_{i=1}^{m}
(f^{(i)}-y^{(i)})
\]

These derivatives measure how changing each parameter affects the cost.

The parameters are updated using:

\[
w_j:=w_j-\alpha\frac{\partial J}{\partial w_j}
\]

\[
b:=b-\alpha\frac{\partial J}{\partial b}
\]

The process repeats:

1. Calculate the predicted probabilities.
2. Calculate the cost across the training dataset.
3. Calculate the gradients with respect to the weights and bias.
4. Update the parameters using the learning rate.
5. Repeat until the optimization converges or a stopping criterion is reached.

The important connection is that the loss function defines how individual predictions are evaluated, the cost function combines those losses into an objective, and gradient descent uses the derivatives of that objective to improve the model.

## 10. Practical Implementation Considerations

When implementing logistic regression, several details help ensure that the cost function behaves correctly.

**Numerical stability:** Directly evaluating logarithms at probabilities of exactly 0 or 1 can cause numerical errors. Implementations should use numerically stable functions, such as a logits-based binary cross-entropy operation, instead of calculating probabilities and logarithms naively when possible.

**Vectorization:** The cost can be computed for an entire dataset using array operations rather than explicit loops. This is generally more efficient and makes it easier to work with large datasets.

**Feature scaling:** Scaling features to comparable ranges can improve the convergence speed of gradient descent. It does not change the fundamental convexity property of standard logistic regression.

**Regularization:** An additional penalty can be added to the cost function to discourage excessively large weights and reduce overfitting. Standard L2 regularization adds a term proportional to the sum of squared weights. The resulting objective remains convex for ordinary logistic regression with L2 regularization.

**Evaluation metrics:** A low training cost does not automatically guarantee good generalization. Performance should also be assessed using suitable validation data and metrics such as precision, recall, accuracy, and ROC-AUC, depending on the classification problem.

## 11. Summary

Logistic regression requires a loss function that evaluates predicted probabilities rather than simply measuring squared differences between predictions and labels.

Squared error, although effective in linear regression, can produce a non-convex objective when combined with the sigmoid function. Logistic loss provides a more suitable alternative by assigning low loss to accurate predictions and sharply penalizing confident incorrect predictions.

The individual loss is:

\[
L(f(x),y)=-y\log(f(x))-(1-y)\log(1-f(x))
\]

The overall cost is the average loss across the training examples:

\[
J(w,b)=\frac{1}{m}\sum_{i=1}^{m}L(f_{w,b}(x^{(i)}),y^{(i)})
\]

This cost function is convex in the standard logistic regression model, supporting reliable optimization toward a global minimum when the relevant convergence conditions are satisfied. It is also derived from maximum likelihood estimation, connecting the training objective directly to probability theory.

The overall learning process follows a clear sequence:

**Input features → Linear combination → Sigmoid probability → Logistic loss → Average cost → Gradient descent → Updated parameters.**

Understanding this sequence establishes the foundation for implementing logistic regression, interpreting its optimization process, and extending the model with regularization and more advanced training techniques.